In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# =============================================================================
# STAGE 0 — DATASET INGEST                  (Drive must already be mounted)
#
# Builds the canonical dataset that every later stage reads from:
#
#   derived/meta/strokes.parquet    1457 strokes, corrected, classed, foldeda
#   derived/meta/events.parquet     bounce / net / empty_event (for Stage 6)
#   derived/meta/rallies.parquet    rally-ending outcomes (for Phase 2)
#   derived/meta/folds.json         the 7 grouped LOVO folds
#
# Applies four source-annotation corrections found in the Step 1 audit, and
# validates hard invariants. Fails loudly rather than writing bad data.
#
# Runs in seconds. Touches no video.
# =============================================================================

BASE = "/content/drive/MyDrive/tt_coach"

# --- window geometry (single source of truth for Stages 2-5) -----------------
NATIVE_FPS   = 120
PRE_FRAMES   = 60      # 0.50 s before contact — captures the backswing
POST_FRAMES  = 36      # 0.30 s after contact  — captures the follow-through
# Asymmetric on purpose: a block's defining property is the ABSENCE of a
# backswing, so pre-contact carries more class information than follow-through.

import json, re, sys
from pathlib import Path
from collections import Counter

import pandas as pd
import yaml

BASE = Path(BASE)
LABELS = BASE / "raw/labels_extended"
META = BASE / "derived/meta"
META.mkdir(parents=True, exist_ok=True)

VIDEOS = [f"game_{i}" for i in range(1, 6)] + [f"test_{i}" for i in range(1, 8)]

# =============================================================================
# Source-annotation corrections (from the Step 1 audit)
# Applied here, never by editing the source files — the raw data stays pristine
# and every correction is auditable in one place.
# =============================================================================
PRIMARY_FIX = {
    "xright_backhand_chop": "right_backhand_chop",   # stray 'x' typo
}
LEAN_FIX = {
    "back_heavyn": "back_heavy",                     # trailing 'n' typo
}
DROP_PRIMARY = {
    "point",        # Step-1 placeholder the annotators never replaced
}

# =============================================================================
# The 7 grouped folds.
# Single-video LOVO leaves four folds with a class at zero support (test_2 has
# no control, test_3/test_5 no defence), which makes macro-F1 on those folds
# meaningless. Grouping the thin test videos gives every fold all four classes
# and >=150 strokes, while still guaranteeing no video spans train and val.
# =============================================================================
FOLDS = {
    "A": ["game_1"],
    "B": ["game_2"],
    "C": ["game_3"],
    "D": ["game_4"],
    "E": ["game_5"],
    "F": ["test_1", "test_4"],
    "G": ["test_2", "test_3", "test_5", "test_6", "test_7"],
}
VIDEO2FOLD = {v: f for f, vs in FOLDS.items() for v in vs}

EXPECTED_STROKES = 1457
EXPECTED_TECH = {"loop": 583, "serve": 290, "push": 279, "block": 187,
                 "flick": 65, "chop": 31, "smash": 12, "lob": 10}

# --- taxonomy -----------------------------------------------------------------
tax_path = BASE / "TAXONOMY.yaml"
if not tax_path.exists():
    sys.exit(f"TAXONOMY.yaml missing at {tax_path} — please ensure Drive is mounted correctly.")

TAX = yaml.safe_load(tax_path.read_text())
TECH2CLASS = {t: c for c, s in TAX["classes"].items() for t in s["techniques"]}
SIDES = TAX["label_format"]["side_values"]
HIGHS = TAX["label_format"]["high_level_values"]
TECHS = TAX["label_format"]["technique_values"]
STROKE_RE = re.compile(
    rf"^({'|'.join(SIDES)})_({'|'.join(HIGHS)})_({'|'.join(TECHS)})$ ")

RALLY_RE = re.compile(
    r"^(left|right)_(out|net|winner|not_hitting_ball|double_bounce|"
    r"miss_on_own_side)$")
PLAIN_EVENTS = {"bounce", "net", "empty_event"}

# --- video lengths ------------------------------------------------------------
vm_path = META / "video_manifest.csv"
if not vm_path.exists():
    sys.exit("video_manifest.csv missing — run STEP 2 first.")
vm = pd.read_csv(vm_path).set_index("video_id")
NFRAMES = vm["nb_frames"].to_dict()

ValueError: Mountpoint must not already contain files